In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
%pip install langchain
%pip install faiss-cpu
%pip install sentence-transformers
%pip install pypdf
%pip install transformers
%pip install accelerate
%pip install InstructorEmbedding


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: joblib>=0.11 in c:\users\arigilasrinivas\anaconda3\lib\site-packages (from scikit-learn->sentence-transformers) (0.16.0)



In [3]:
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.embeddings import HuggingFaceInstructEmbeddings
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline

from transformers import pipeline

In [4]:
my_file = PyPDFDirectoryLoader("pdfs")

In [5]:
data = my_file.load()

In [6]:
data[0]

Document(metadata={'source': 'pdfs\\YOLOv8_ State-of-the-Art Computer Vision Model.pdf', 'page': 0}, page_content='3/4/26, 11:58 PM YOLOv8: State-of-the-Art Computer Vision Model\nhttps://yolov8.com 1/5\nWhat is YOLOv8?\nYOLOv8 is a computer vision model architecture developed by Ultralytics, the creators of YOLOv5.\nY ou can deploy YOLOv8 models on a wide range of devices, including NVIDIA  Jetson, NVIDIA\nGPUs, and macOS systems with Roboflow Inference, an open source Python package for running\nvision models.\nGet Started Using YOLOv8\nRoboflow is the fastest way to get YOLOv8 running in production. Manage dataset versioning, preprocessing, augmentation,\ntraining, evaluation, and deployment all in one workflow . Easily upload data, train YOLOv8 with best-practice defaults,\ncompare runs, and deploy to edge, cloud, or API in minutes. T ry a  YOLOv8 model  on Roboflow with this workflow:\nYOLOv5 YOLOv8 YOLO1 1\nP y t h o n c U R L J a v a s c r i p t S w i f t . N e t\nfrom inference

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)

In [8]:
text_chunks = text_splitter.split_documents(data)

In [9]:
len(text_chunks)

16

In [10]:
print(text_chunks[0])

page_content='3/4/26, 11:58 PM YOLOv8: State-of-the-Art Computer Vision Model
https://yolov8.com 1/5
What is YOLOv8?
YOLOv8 is a computer vision model architecture developed by Ultralytics, the creators of YOLOv5.
Y ou can deploy YOLOv8 models on a wide range of devices, including NVIDIA  Jetson, NVIDIA
GPUs, and macOS systems with Roboflow Inference, an open source Python package for running
vision models.
Get Started Using YOLOv8' metadata={'source': 'pdfs\\YOLOv8_ State-of-the-Art Computer Vision Model.pdf', 'page': 0}


In [11]:
print(text_chunks[0].page_content)

3/4/26, 11:58 PM YOLOv8: State-of-the-Art Computer Vision Model
https://yolov8.com 1/5
What is YOLOv8?
YOLOv8 is a computer vision model architecture developed by Ultralytics, the creators of YOLOv5.
Y ou can deploy YOLOv8 models on a wide range of devices, including NVIDIA  Jetson, NVIDIA
GPUs, and macOS systems with Roboflow Inference, an open source Python package for running
vision models.
Get Started Using YOLOv8


In [12]:
# option 1 is to use HuggingfaceEmbeddings & Sentence Transformers
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


## option 2 is to use BAAI - BGE

# embeddings = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-small-en",
#     model_kwargs={"device": "cpu"},
#     encode_kwargs={"normalize_embeddings": True}
# )

# option 3 is to use HuggingfaceInstructEmbeddings
# embeddings = HuggingFaceInstructEmbeddings(model_name="hkunlp/instructor-base")


In [14]:
docsearch = FAISS.from_documents(text_chunks, embeddings)

In [15]:
query = "which models does the Yolov8 outperform"

In [16]:
docs = docsearch.similarity_search(query)

In [17]:
## the output here would be in the form of vector
docs

In [18]:
## we need to get output in form of text, so we will use LLM and Retrieval QA

In [19]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base", # google/flan-t5-small -> if laptop is too slow
    max_length=256
)

llm = HuggingFacePipeline(pipeline=generator)

<ipython-input-19-112a7ad0b6fe>:7: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  llm = HuggingFacePipeline(pipeline=generator)


In [21]:
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=docsearch.as_retriever())

In [22]:
query = "which models does the Yolov8 outperform"

In [23]:
# this model will now generate correct output , v8 outperforms v7, v6, v5 etc
qa.run(query)

In [24]:
import sys

while True:
    user_input = input("Input Prompt: ")

    if user_input == "exit":
        print("Exiting..!!")
        sys.exit()

    if user_input == "":
        continue

    result = qa.run(user_input)

    print("Answer:", result)

In [26]:
#### IN THIS APPROACH WE ARE USING AUTO-TOKENIZER AND AUTO-MODEL and MANUALLY CREATING EMBEDDINGS ###

In [25]:
## Manually creating embeddings

from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en")
model = AutoModel.from_pretrained("BAAI/bge-small-en")

def embed(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)

    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.numpy()

In [ ]:
# text_chunks

In [29]:
texts = [doc.page_content for doc in text_chunks]

In [30]:
embeddings_list = [embed(text) for text in texts]

In [31]:
import numpy as np

embeddings_matrix = np.vstack(embeddings_list)

In [32]:
import faiss

dimension = embeddings_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings_matrix)

In [33]:
documents = texts

In [34]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base", # google/flan-t5-small -> if laptop is too slow
    max_length=256
)

llm = HuggingFacePipeline(pipeline=generator)

In [ ]:
# query = "Which models does YOLOv8 outperform?"

# query_vector = embed(query)

In [ ]:
# k = 3

# distances, indices = index.search(query_vector, k)

In [ ]:
# results = [documents[i] for i in indices[0]]

# for r in results:
#     print(r)

In [ ]:
# context = "\n".join(results)

# prompt = f"""
# Answer the question using the context below.

# Context:
# {context}

# Question:
# {query}
# """

In [ ]:
# response = generator(prompt)

# print(response[0]["generated_text"])

In [35]:
### ***** All the above steps can be wrapped into a function as per below *****

def ask_question(query):

    query_vector = embed(query)

    distances, indices = index.search(query_vector, 3)

    results = [documents[i] for i in indices[0]]

    context = "\n".join(results)

    prompt = f"""
    Answer the question using the context below.

    Context:
    {context}

    Question:
    {query}
    """

    response = generator(prompt)

    return response[0]["generated_text"]

In [36]:
### ask question
ask_question("Which models does YOLOv8 outperform?")

'nano, small, medium, large, and extra-large'

In [37]:
import sys

while True:

    user_input = input("Ask Question: ")

    if user_input == "exit":
        print("Exiting...")
        sys.exit()

    if user_input.strip() == "":
        continue

    answer = ask_question(user_input)

    print("\nAnswer:", answer)
    print()

Ask Question: What is yolov8?

Answer: YOLOv8 is a computer vision model architecture developed by Ultralytics, the creators of YOLOv5.

Ask Question: what is its previous version?

Answer: YOLOv5

Ask Question: How is it better than v5 version?

Answer: Compared to YOLOv8's predecessor, YOLOv5, YOLOv8 comes with: 1. A new anchor-free detection system.2. Changes to the convolutional blocks used in the model.3. Mosaic augmentation applied during training, turned off before the last 10 epochs.

Ask Question: what are the variants in v8?

Answer: nano, small, medium, large, and extra-large

Ask Question: Which one is the best among the above?

Answer: YOLOv8

Ask Question: do we have yolov7 and v6 ?

Answer: yes

Ask Question: how is v8 better than them ?

Answer: RF-DETR is the best alternative to YOLOv8 for object detection and segmentation

Ask Question: primary use-cases of v8?

Answer: RF-DETR

Ask Question: exit
Exiting...


SystemExit: 